In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re
import pandas as pd
from num2words import num2words
from underthesea import text_normalize

In [ ]:
path = "/content/drive/MyDrive/Data_splitting/tsv/dev_en_vi.tsv"
df = pd.read_csv(path, sep="\t")

In [ ]:
df.head()

,audio_path,transcript_en,translation_vi,client_id
0,1191_0001.wav,Pat Mitchell: You have brought us images from ...,Pat Mitchell: Chị đã mang đến các bức ảnh từ T...,1191
1,1191_0002.wav,"Nadia Al - Sakkaf: Well, I'm glad to be here. ...",Nadia Al - Sakkaf: Tôi rất sung sướng được ở đ...,1191
2,1191_0003.wav,This picture shows a revolution started by wom...,Bức ảnh này cho thấy một cuộc cách mạng do phụ...,1191
3,1191_0004.wav,So many people are there. The intensity of the...,Rất nhiều người ở đó. Cuộc nổi dậy rất căng th...,1191
4,1191_0005.wav,These women are learning about first aid and t...,Những phụ nữ này đang học cách sơ cưú và học v...,1191


In [ ]:
emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF"
    "\U0001FA70-\U0001FAFF"
    "]+",
    flags=re.UNICODE
)

def remove_emoji(text):
    if not isinstance(text, str):
        return text
    return emoji_pattern.sub("", text)

def remove_special_chars(text):
    if not isinstance(text, str):
        return text
    return re.sub(r"[^a-zA-Z0-9À-ỹ .,!?'-]", " ", text)
def is_year(num_str):
    try:
        n = int(num_str)
        return 1000 <= n <= 2100
    except:
        return False
def normalize_numbers_en(text):
    if not isinstance(text, str):
        return text

    def convert(match):
        num = match.group()
        if is_year(num):
            return num

        return num2words(num, lang="en")

    return re.sub(r"\d+", convert, text)


def normalize_numbers_vi(text):
    if not isinstance(text, str):
        return text

    def convert(match):
        num = match.group()
        if is_year(num):
            return num

        num = int(num)
        return text_normalize(num2words(num, lang="vi"))

    return re.sub(r"\d+", convert, text)

def clean_en(text):
    if not isinstance(text, str):
        return text
    text = remove_emoji(text)
    text = remove_special_chars(text)
    text = normalize_numbers_en(text)
    text = text.lower()
    return text.strip()


def clean_vi(text):
    if not isinstance(text, str):
        return text
    text = remove_emoji(text)
    text = remove_special_chars(text)
    text = normalize_numbers_vi(text)
    text = text.lower()
    return text.strip()



df["transcript_en"] = df["transcript_en"].apply(clean_en)
df["translation_vi"] = df["translation_vi"].apply(clean_vi)


In [ ]:
df.to_csv("/content/drive/MyDrive/data_small/dev_en_vi.tsv",
                 sep="\t",
                 encoding="utf-8",
                 index=False)